---
## Paso 4: Experimentos

Siguiendo exactamente las instrucciones del PDF:
- **6.1** Impacto del tamaño de lote: 32, 64, 128
- **6.2** Impacto de la tasa de aprendizaje: 1e-2, 1e-3, 1e-4
- **6.3** Impacto del tamaño de la red: filtros 32/64/128 y neuronas 128/256/512

Cada experimento usa el modelo base como referencia y varía **un solo hiperparámetro** a la vez.


### 4.0 Función auxiliar de experimento

In [ ]:
def ejecutar_experimento(config, epocas=15):
    """
    Crea un modelo y optimizador frescos con la configuración dada,
    entrena y retorna el historial.

    Parámetros
    ----------
    config : dict con claves:
        filtros, neuronas, lr, batch_size, nombre
    epocas : int — épocas de entrenamiento
    """
    # Reconstruir pipelines con el batch_size del experimento
    bs = config["batch_size"]
    ds_tr = construir_pipeline(rutas_train, etiq_train, bs,
                               entrenamiento=True)
    ds_vl = construir_pipeline(rutas_val,   etiq_val,   bs,
                               entrenamiento=False)

    # Modelo fresco con la configuración indicada
    modelo_exp = CNN(
        num_clases=NUM_CLASES,
        filtros=config["filtros"],
        neuronas=config["neuronas"],
        tasa_drop=0.4,
        rngs=nnx.Rngs(SEMILLA)
    )

    # Optimizador con el LR indicado
    sched = optax.cosine_decay_schedule(
        init_value=config["lr"],
        decay_steps=epocas * (len(rutas_train) // bs + 1)
    )
    opt_exp = nnx.Optimizer(modelo_exp, optax.adam(sched))

    print(f"\n{'='*55}")
    print(f"  Experimento: {config['nombre']}")
    print(f"{'='*55}")

    hist = entrenar(modelo_exp, opt_exp, ds_tr, ds_vl, epocas)

    # Exactitud en prueba
    p_test, a_test = [], []
    ds_tst = construir_pipeline(rutas_test, etiq_test, bs,
                                entrenamiento=False)
    for imgs_tf, etiq_tf in ds_tst:
        imgs = jnp.array(imgs_tf.numpy())
        etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
        p, a = paso_evaluacion(modelo_exp, imgs, etiq)
        p_test.append(float(p)); a_test.append(float(a))

    hist["exactitud_prueba"] = float(np.mean(a_test))
    hist["perdida_prueba"]   = float(np.mean(p_test))
    hist["config"]           = config
    hist["throughput"]       = len(rutas_train) / hist["t_media_epoca"]

    print(f"  Exactitud prueba : {hist['exactitud_prueba']*100:.2f}%")
    return hist, modelo_exp


EPOCAS_EXP = 15   # épocas por experimento (menos que el base para ahorrar tiempo)
print(f"Función ejecutar_experimento() lista — {EPOCAS_EXP} épocas por experimento")


---
### 6.1 Experimento: Impacto del Tamaño de Lote

Se mantienen fijos: LR=1e-3, filtros=32, neuronas=256  
Se varía: **batch_size ∈ {32, 64, 128}**


In [ ]:
resultados_batch = {}

configs_batch = [
    {"nombre": "batch=32",  "batch_size": 32,  "lr": 1e-3, "filtros": 32, "neuronas": 256},
    {"nombre": "batch=64",  "batch_size": 64,  "lr": 1e-3, "filtros": 32, "neuronas": 256},
    {"nombre": "batch=128", "batch_size": 128, "lr": 1e-3, "filtros": 32, "neuronas": 256},
]

for cfg in configs_batch:
    hist, _ = ejecutar_experimento(cfg, epocas=EPOCAS_EXP)
    resultados_batch[cfg["nombre"]] = hist


In [ ]:
# ── Tabla comparativa batch ───────────────────────────────────────────────
print(f"{'Config':<12} {'T.Total(s)':>10} {'T/época(s)':>11} "
      f"{'Acc.Val':>9} {'Acc.Test':>9} {'Throughput':>12}")
print("-" * 66)
for nombre, h in resultados_batch.items():
    print(f"{nombre:<12} {h['tiempo_total']:>10.1f} {h['t_media_epoca']:>11.1f} "
          f"{h['exactitud_final_val']*100:>8.2f}% "
          f"{h['exactitud_prueba']*100:>8.2f}% "
          f"{h['throughput']:>10.0f} img/s")


In [ ]:
# ── Gráfica batch ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colores = {"batch=32": "#2563EB", "batch=64": "#16A34A", "batch=128": "#D97706"}

for nombre, h in resultados_batch.items():
    ep = range(1, len(h["perdida_val"]) + 1)
    axes[0].plot(ep, h["perdida_val"],   "o-", color=colores[nombre],
                 lw=2, ms=4, label=nombre)
    axes[1].plot(ep, [v*100 for v in h["exactitud_val"]], "o-",
                 color=colores[nombre], lw=2, ms=4, label=nombre)

for ax, titulo, ylabel in zip(axes,
    ["Pérdida Validación", "Exactitud Validación (%)"],
    ["Pérdida", "Exactitud (%)"]):
    ax.set_xlabel("Época"); ax.set_ylabel(ylabel)
    ax.set_title(titulo, fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_facecolor("#F8FAFC")

fig.suptitle("Experimento 6.1 — Impacto del Tamaño de Lote",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


---
### 6.2 Experimento: Impacto de la Tasa de Aprendizaje

Se mantienen fijos: batch=32, filtros=32, neuronas=256  
Se varía: **lr ∈ {1e-2, 1e-3, 1e-4}**


In [ ]:
resultados_lr = {}

configs_lr = [
    {"nombre": "lr=1e-2", "batch_size": 32, "lr": 1e-2, "filtros": 32, "neuronas": 256},
    {"nombre": "lr=1e-3", "batch_size": 32, "lr": 1e-3, "filtros": 32, "neuronas": 256},
    {"nombre": "lr=1e-4", "batch_size": 32, "lr": 1e-4, "filtros": 32, "neuronas": 256},
]

for cfg in configs_lr:
    hist, _ = ejecutar_experimento(cfg, epocas=EPOCAS_EXP)
    resultados_lr[cfg["nombre"]] = hist


In [ ]:
# ── Tabla comparativa LR ──────────────────────────────────────────────────
print(f"{'Config':<10} {'T.Total(s)':>10} {'T/época(s)':>11} "
      f"{'Acc.Val':>9} {'Acc.Test':>9} {'Convergencia':>14}")
print("-" * 66)
for nombre, h in resultados_lr.items():
    # Época en que supera 50% de validación
    conv = next((i+1 for i,v in enumerate(h["exactitud_val"]) if v >= 0.50), None)
    conv_str = f"época {conv}" if conv else "no alcanzó"
    print(f"{nombre:<10} {h['tiempo_total']:>10.1f} {h['t_media_epoca']:>11.1f} "
          f"{h['exactitud_final_val']*100:>8.2f}% "
          f"{h['exactitud_prueba']*100:>8.2f}% "
          f"{conv_str:>14}")


In [ ]:
# ── Gráfica LR ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colores_lr = {"lr=1e-2": "#DC2626", "lr=1e-3": "#2563EB", "lr=1e-4": "#16A34A"}

for nombre, h in resultados_lr.items():
    ep = range(1, len(h["perdida_val"]) + 1)
    axes[0].plot(ep, h["perdida_val"],   "o-", color=colores_lr[nombre],
                 lw=2, ms=4, label=nombre)
    axes[1].plot(ep, [v*100 for v in h["exactitud_val"]], "o-",
                 color=colores_lr[nombre], lw=2, ms=4, label=nombre)

for ax, titulo, ylabel in zip(axes,
    ["Pérdida Validación", "Exactitud Validación (%)"],
    ["Pérdida", "Exactitud (%)"]):
    ax.set_xlabel("Época"); ax.set_ylabel(ylabel)
    ax.set_title(titulo, fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_facecolor("#F8FAFC")

fig.suptitle("Experimento 6.2 — Impacto de la Tasa de Aprendizaje",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


---
### 6.3 Experimento: Impacto del Tamaño de la Red

Se mantienen fijos: batch=32, lr=1e-3  
Se varía: **filtros ∈ {32, 64, 128}** y **neuronas ∈ {128, 256, 512}**


In [ ]:
resultados_red = {}

configs_red = [
    {"nombre": "f32-n128",  "batch_size": 32, "lr": 1e-3, "filtros": 32,  "neuronas": 128},
    {"nombre": "f64-n256",  "batch_size": 32, "lr": 1e-3, "filtros": 64,  "neuronas": 256},
    {"nombre": "f128-n512", "batch_size": 32, "lr": 1e-3, "filtros": 128, "neuronas": 512},
]

for cfg in configs_red:
    hist, _ = ejecutar_experimento(cfg, epocas=EPOCAS_EXP)
    resultados_red[cfg["nombre"]] = hist


In [ ]:
# ── Tabla comparativa tamaño de red ──────────────────────────────────────
print(f"{'Config':<12} {'Parámetros':>12} {'T/época(s)':>11} "
      f"{'Acc.Val':>9} {'Acc.Test':>9}")
print("-" * 58)

params_ref = {
    "f32-n128":  32*2*3*3 + 64*2*3*3 + 128*2*3*3 + 8*8*128*128 + 128*12,
    "f64-n256":  64*2*3*3 + 128*2*3*3 + 256*2*3*3 + 8*8*256*256 + 256*12,
    "f128-n512": 128*2*3*3 + 256*2*3*3 + 512*2*3*3 + 8*8*512*512 + 512*12,
}

for nombre, h in resultados_red.items():
    print(f"{nombre:<12} {'~'+str(params_ref.get(nombre,'?')):>12} "
          f"{h['t_media_epoca']:>11.1f} "
          f"{h['exactitud_final_val']*100:>8.2f}% "
          f"{h['exactitud_prueba']*100:>8.2f}%")


In [ ]:
# ── Gráfica tamaño de red ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colores_red = {"f32-n128": "#16A34A", "f64-n256": "#2563EB", "f128-n512": "#7C3AED"}

for nombre, h in resultados_red.items():
    ep = range(1, len(h["perdida_val"]) + 1)
    lbl = nombre.replace("f","filtros=").replace("-n"," · neuronas=")
    axes[0].plot(ep, h["perdida_val"],   "o-", color=colores_red[nombre],
                 lw=2, ms=4, label=lbl)
    axes[1].plot(ep, [v*100 for v in h["exactitud_val"]], "o-",
                 color=colores_red[nombre], lw=2, ms=4, label=lbl)

for ax, titulo, ylabel in zip(axes,
    ["Pérdida Validación", "Exactitud Validación (%)"],
    ["Pérdida", "Exactitud (%)"]):
    ax.set_xlabel("Época"); ax.set_ylabel(ylabel)
    ax.set_title(titulo, fontweight="bold")
    ax.legend(fontsize=8.5); ax.grid(True, alpha=0.3)
    ax.set_facecolor("#F8FAFC")

fig.suptitle("Experimento 6.3 — Impacto del Tamaño de la Red",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


---
### Resumen global de experimentos

In [ ]:
print("=" * 70)
print("  RESUMEN GLOBAL DE EXPERIMENTOS")
print("=" * 70)

todas = {}
todas.update(resultados_batch)
todas.update(resultados_lr)
todas.update(resultados_red)

print(f"\n{'Experimento':<14} {'Acc.Val':>9} {'Acc.Test':>9} "
      f"{'T/época':>9} {'Throughput':>12}")
print("-" * 58)
for nombre, h in todas.items():
    print(f"{nombre:<14} "
          f"{h['exactitud_final_val']*100:>8.2f}% "
          f"{h['exactitud_prueba']*100:>8.2f}% "
          f"{h['t_media_epoca']:>8.1f}s "
          f"{h['throughput']:>10.0f} img/s")

# Mejor configuración
mejor = max(todas.items(), key=lambda x: x[1]["exactitud_prueba"])
print()
print(f"  Mejor configuración : {mejor[0]}")
print(f"  Exactitud prueba    : {mejor[1]['exactitud_prueba']*100:.2f}%")
print(f"  Exactitud val       : {mejor[1]['exactitud_final_val']*100:.2f}%")
print("=" * 70)
print("\n  Paso 4 completado — listos para optimización de hiperparámetros.")
